# Exploratory Analysis: Accessible Decoded Neurofeedback

This notebook provides an introductory walkthrough of the key components of the
multimodal neurofeedback framework:

1. **fMRI decoding** with MVPA and deep learning
2. **EEG feature extraction** and classification
3. **fNIRS feature extraction** and classification
4. **Cross-modal representation mapping** (CCA and contrastive deep learning)
5. **Closed-loop RL** in a simulated neurofeedback environment
6. **Perception and confidence tasks** in simulation mode

All cells use **simulated data** so no neuroimaging hardware is required.

In [ ]:
import sys
import os

# Add repo root to path if running from notebooks directory
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import matplotlib.pyplot as plt

print('Framework loaded successfully.')

## 1. fMRI Neural Decoding

In [ ]:
from models.fmri_decoder import FMRIDecoder

rng = np.random.default_rng(42)

# Simulate fMRI voxel patterns: 200 trials, 500 voxels, 2 classes
n_trials, n_voxels, n_classes = 200, 500, 2
X = rng.normal(size=(n_trials, n_voxels))
y = rng.integers(0, n_classes, size=n_trials)

# Add a discriminative signal for class 1
X[y == 1, :50] += 0.8

split = int(0.8 * n_trials)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# --- MVPA decoder ---
mvpa = FMRIDecoder(backend='mvpa', estimator='svm', C=1.0)
mvpa.fit(X_train, y_train)
print(f'MVPA accuracy: {mvpa.score(X_test, y_test):.2%}')

# --- Deep learning decoder ---
deep = FMRIDecoder(backend='deep', n_voxels=n_voxels, n_classes=n_classes, epochs=10)
deep.fit(X_train, y_train)
print(f'Deep decoder accuracy: {deep.score(X_test, y_test):.2%}')

## 2. EEG Feature Extraction and Decoding

In [ ]:
from models.eeg_decoder import EEGDecoder

sfreq = 256.0
n_epochs, n_channels, n_times = 100, 8, int(sfreq * 2)  # 2-second epochs

# Simulate EEG epochs
epochs = rng.normal(size=(n_epochs, n_channels, n_times))
y_eeg = rng.integers(0, 2, size=n_epochs)

# Add alpha-band signal for class 1
t = np.arange(n_times) / sfreq
epochs[y_eeg == 1, 0, :] += 3.0 * np.sin(2 * np.pi * 10 * t)

split = int(0.8 * n_epochs)
decoder_eeg = EEGDecoder(sfreq=sfreq)
decoder_eeg.fit(epochs[:split], y_eeg[:split])
print(f'EEG decoder accuracy: {decoder_eeg.score(epochs[split:], y_eeg[split:]):.2%}')

## 3. fNIRS Feature Extraction and Decoding

In [ ]:
from models.fnirs_decoder import FNIRSDecoder

sfreq_nirs = 10.0
n_ep, n_ch, n_t = 80, 4, int(sfreq_nirs * 20)  # 20-second epochs

hbo = rng.normal(scale=1e-6, size=(n_ep, n_ch, n_t))
hbr = rng.normal(scale=1e-6, size=(n_ep, n_ch, n_t))
y_nirs = rng.integers(0, 2, size=n_ep)

# Add HbO increase for class 1
hbo[y_nirs == 1, 0, 50:150] += 2e-6

split = int(0.8 * n_ep)
decoder_nirs = FNIRSDecoder(sfreq=sfreq_nirs, tmin=0.0, tmax=20.0)
decoder_nirs.fit(hbo[:split], hbr[:split], y_nirs[:split])
print(f'fNIRS decoder accuracy: {decoder_nirs.score(hbo[split:], hbr[split:], y_nirs[split:]):.2%}')

## 4. Cross-Modal Representation Mapping

In [ ]:
from cross_modal import CrossModalMapper

n_samples = 150
fmri_feats = rng.normal(size=(n_samples, 20))   # simulated fMRI features
eeg_feats  = rng.normal(size=(n_samples, 40))   # simulated EEG features

# Add shared structure: a shared latent variable drives both
latent = rng.normal(size=(n_samples, 5))
fmri_feats[:, :5] += 2.0 * latent
eeg_feats[:,  :5] += 2.0 * latent

mapper = CrossModalMapper(method='cca', n_components=5)
mapper.fit(fmri_feats, eeg_feats)
fmri_canon, eeg_canon = mapper.transform(fmri_feats, eeg_feats)

corrs = [np.corrcoef(fmri_canon[:, i], eeg_canon[:, i])[0, 1] for i in range(5)]
print('Canonical correlations:', [f'{c:.3f}' for c in corrs])

## 5. Closed-Loop RL Neurofeedback

In [ ]:
from closed_loop import RLFeedbackAgent
from closed_loop.reinforcement_learning_feedback import NeurofeedbackEnv, ActorCriticAgent

env = NeurofeedbackEnv(n_steps=20, target_similarity=0.65)

# REINFORCE agent
reinforce = RLFeedbackAgent(obs_dim=2, n_actions=5)
rewards_reinforce = reinforce.train(env, n_episodes=300)

# Actor-Critic agent
ac = ActorCriticAgent(obs_dim=2, n_actions=5)
rewards_ac = ac.train(env, n_episodes=300)

fig, ax = plt.subplots(figsize=(9, 4))

window = 20
for name, rewards in [('REINFORCE', rewards_reinforce), ('Actor-Critic', rewards_ac)]:
    smoothed = np.convolve(rewards, np.ones(window) / window, mode='valid')
    ax.plot(smoothed, label=name)

ax.set_xlabel('Episode')
ax.set_ylabel('Total reward (smoothed)')
ax.set_title('RL Agent Training Curves')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Experimental Tasks in Simulation Mode

In [ ]:
from experiments import PerceptionTask, ConfidenceTask

# Perception task
perc = PerceptionTask(n_trials=30, with_feedback=True, random_state=42)
perc.run_simulation()
perc.report()

# Confidence task
conf = ConfidenceTask(n_trials=40, dprime=1.5, random_state=42)
conf.run_simulation()
conf.report()

## 7. Full Pipeline Simulation

In [ ]:
from pipelines import RealtimeNeurofeedbackPipeline
from models.fmri_decoder import FMRIDecoder
from closed_loop.reinforcement_learning_feedback import ActorCriticAgent

# Build and train pipeline components
decoder = FMRIDecoder(backend='mvpa', estimator='svm')
agent   = ActorCriticAgent(obs_dim=2, n_actions=5)

pipeline = RealtimeNeurofeedbackPipeline(
    decoder=decoder,
    agent=agent,
    verbose=True,
)

# Train decoder on simulated localiser data
X_loc = rng.normal(size=(100, 500))
y_loc = rng.integers(0, 2, size=100)
X_loc[y_loc == 1, :50] += 0.8

pipeline.train_decoder(X_loc, y_loc)
pipeline.pretrain_agent(n_episodes=200)

# Run simulation session
log = pipeline.run_simulation(n_trials=10)

similarities = [entry['similarity'] for entry in log]
plt.figure(figsize=(8, 3))
plt.plot(similarities, marker='o')
plt.axhline(0.65, color='r', linestyle='--', label='Target')
plt.xlabel('Trial')
plt.ylabel('Decoded similarity')
plt.title('Simulated Closed-Loop Session')
plt.legend()
plt.tight_layout()
plt.show()